# 🧪 AI-Scientist-v2 on Kaggle

**Fully Automated Scientific Discovery via Agentic Tree Search**  
*Powered by DeepSeek V4 Flash 🚀*

This notebook runs [AI-Scientist-v2](https://github.com/SakanaAI/AI-Scientist_v2) by SakanaAI on Kaggle's free GPU (NVIDIA T4).

### Pipeline Overview:
1. **Setup** — Clone repo, install dependencies
2. **Ideation** — Generate research ideas via LLM
3. **Experimentation** — Agentic tree search runs experiments
4. **Write-up** — Generate a scientific paper with LaTeX
5. **Review** — LLM self-review of the paper

> ⚠️ **Warning:** This code executes LLM-written code. Run in a sandboxed environment.
> 
> 💰 **Cost:** ~$1-3 per run with **DeepSeek V4 Flash** (very affordable!). Kaggle GPU is **free**.

---
## 1. ⚙️ Setup Environment

Run the cell below to:
- Clone the AI-Scientist-v2 repository
- Install system packages (LaTeX, poppler, chktex)
- Install Python dependencies
- Verify GPU availability

In [ ]:
import os, sys, subprocess, json, shutil, time
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
PROJECT_DIR = WORKING_DIR / "AI-Scientist-v2"
os.chdir(WORKING_DIR)

print("=" * 60)
print("🚀 Starting AI-Scientist-v2 Setup on Kaggle")
print("=" * 60)

# ── Clone repo (your fork with DeepSeek V4 Flash support) ──
REPO_URL = "https://github.com/brahimje/AI-Scientist-v2"

if not PROJECT_DIR.exists():
    print(f"\n[1/6] Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print("\n[1/6] Repository already exists, pulling latest...")
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}

# ── Install system packages (minimal LaTeX for speed) ──
print("\n[2/6] Installing system packages (minimal LaTeX, poppler)...")
import subprocess as sp
import os
env = os.environ.copy()
env["DEBIAN_FRONTEND"] = "noninteractive"

cmds = [
    "apt-get update -qq",
    "apt-get install -y -qq --no-install-recommends texlive-latex-base texlive-latex-extra texlive-fonts-recommended texlive-bibtex-extra poppler-utils cm-super 2>&1",
]
for cmd in cmds:
    result = sp.run(cmd, shell=True, env=env, capture_output=True, text=True, timeout=600)
    if result.returncode != 0:
        print(f"   ⚠️ Warning (non-fatal): {result.stderr[-200:]}")
    else:
        print(f"   ✅ Package installed")

print("   ✅ System packages ready")

# ── Install Python dependencies ──
print("\n[3/6] Installing Python dependencies...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install -q -r requirements.txt
print("    ✅ Python packages installed")

# ── Verify GPU ──
print("\n[4/6] Verifying GPU...")
import torch
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    for i in range(gpu_count):
        print(f"    ✅ GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"       VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
else:
    print("    ⚠️ No GPU detected. CPU-only mode (slow for experiments).")

# ── Verify Python ──
print(f"\n[5/6] Python version: {sys.version}")

# ── Create directories ──
print("\n[6/6] Creating project directories...")
os.makedirs(PROJECT_DIR / "ai_scientist" / "ideas", exist_ok=True)
os.makedirs(PROJECT_DIR / "experiments", exist_ok=True)
os.makedirs(PROJECT_DIR / "logs", exist_ok=True)
print("    ✅ Directories created")

print("\n" + "=" * 60)
print("✅ Setup complete!")
print("=" * 60)

---
## 2. 🔑 Configure API Keys

Set your API keys as **Kaggle Secrets** (`Add-ons > Secrets` in the sidebar):

| Secret Name | Description | Required? |
|---|---|---|
| `DEEPSEEK_API_KEY` | DeepSeek API key (for `deepseek-v4-flash`) | ✅ **Yes** |
| `ANDROZOO_API_KEY` | AndroZoo dataset API key | ✅ **Yes** (for Android malware data) |
| `S2_API_KEY` | Semantic Scholar API key (for literature search) | ⚠️ Optional but recommended |
| `OPENAI_API_KEY` | OpenAI API key (GPT-4o, o1, o3) | ⚠️ Optional (fallback) |

> 💡 **DeepSeek V4 Flash** is ~10x cheaper than GPT-4o!  
> Get your DeepSeek key: [https://platform.deepseek.com/api_keys](https://platform.deepseek.com/api_keys)  
> Get a **free** Semantic Scholar key: [https://www.semanticscholar.org/product/api](https://www.semanticscholar.org/product/api)

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# ── Load API Keys from Kaggle Secrets ──
api_keys = {
    "DEEPSEEK_API_KEY": None,
    "ANDROZOO_API_KEY": None,
    "S2_API_KEY": None,
    "OPENAI_API_KEY": None,
    "ANTHROPIC_API_KEY": None,
}

for key in api_keys:
    try:
        val = secrets.get_secret(key)
        if val:
            os.environ[key] = val
            api_keys[key] = "✅ Set (hidden)"
            print(f"    ✅ {key} loaded from Kaggle Secrets")
        else:
            api_keys[key] = "❌ Empty"
            print(f"    ⚠️ {key} is empty")
    except:
        api_keys[key] = "❌ Not set"
        print(f"    ⚠️ {key} not found in Kaggle Secrets")

print("\n" + "─" * 40)
if api_keys["DEEPSEEK_API_KEY"]:
    print("✅ DeepSeek API key is set! Ready to use deepseek-v4-flash.")
else:
    print("⚠️ Please add DEEPSEEK_API_KEY as a Kaggle Secret, then re-run this cell.")
    print("   Get one at: https://platform.deepseek.com/api_keys")

---
## 3. 💡 Generate Research Ideas (Ideation)

Create a topic description file and run the ideation script to generate research ideas.

**Edit the topic below** to match your research area of interest!

In [ ]:
%cd {PROJECT_DIR}

# ── Choose which idea to use ──
# Option A: Use the built-in example idea (simpler, higher success rate)
# Option B: Create your custom topic below
USE_EXAMPLE_IDEA = False  # Set to True for built-in ICBINB example

if USE_EXAMPLE_IDEA:
    print("✅ Using built-in example idea: ICBINB (simpler, higher success rate)")
    print("   To use your own topic, set USE_EXAMPLE_IDEA = False above")
else:
    TOPIC_TITLE = "AI-Powered Android Malware Detection via Static and Dynamic Analysis"
    TOPIC_KEYWORDS = "android security, malware detection, static analysis, dynamic analysis, deep learning, mobile security"
    TOPIC_TLDR = "Combining static APK analysis and dynamic runtime behavior monitoring with lightweight neural networks for real-time Android malware detection."
    TOPIC_ABSTRACT = (
        "Android devices are increasingly targeted by sophisticated malware that evades "
        "traditional signature-based detection. This work investigates deep learning approaches "
        "for Android malware detection that combine static analysis of APK files (permissions, "
        "API calls, intent filters) with dynamic analysis of runtime behaviors (system calls, "
        "network activity, battery usage patterns). We explore lightweight neural architectures "
        "suitable for on-device or edge deployment, including 1D CNNs for permission sequences, "
        "graph neural networks for API call graphs, and temporal models for system call traces. "
        "The key hypothesis is that a hybrid static-dynamic fusion model can achieve higher "
        "detection accuracy than either approach alone, while maintaining inference speeds "
        "suitable for real-time protection. We evaluate on standard Android malware datasets "
        "(Drebin, CICAndMal2017, AndroZoo) and benchmark against signature-based tools, "
        "traditional ML classifiers, and existing deep learning methods. Experiments measure "
        "detection F1-score, false positive rate, inference latency, and robustness to "
        "common evasion techniques including obfuscation and repackaging."
    )
    
    topic_md = f"""# Title: {TOPIC_TITLE}

## Keywords
{TOPIC_KEYWORDS}

## TL;DR
{TOPIC_TLDR}

## Abstract
{TOPIC_ABSTRACT}
"""

    topic_file = PROJECT_DIR / "ai_scientist" / "ideas" / "my_research_topic.md"
    topic_file.write_text(topic_md)
    
    print(f"✅ Created topic: {TOPIC_TITLE}")
    print(f"📌 Keywords: {TOPIC_KEYWORDS}")

In [ ]:
%cd {PROJECT_DIR}

# ── Run Ideation ──
# This generates research ideas using an LLM
# ⏱️ Expected time: ~10-30 minutes depending on model

IDEATION_MODEL = "deepseek-v4-flash"  # 🚀 Fast & cheap! Use "gpt-4o-mini" as fallback.
NUM_IDEAS = 3                         # Number of ideas to generate (start small for testing)
NUM_REFLECTIONS = 3                   # Refinement steps per idea

# Use the correct topic file based on your choice above
if USE_EXAMPLE_IDEA:
    topic = "ai_scientist/ideas/i_cant_believe_its_not_better.md"
    ideas_out = "ai_scientist/ideas/i_cant_believe_its_not_better.json"
    print("📌 Using example topic: ICBINB")
else:
    topic = "ai_scientist/ideas/my_research_topic.md"
    ideas_out = "ai_scientist/ideas/my_research_topic.json"
    print("📌 Using custom topic")

print(f"🚀 Running ideation with {IDEATION_MODEL}...")
print(f"   Generating {NUM_IDEAS} ideas with {NUM_REFLECTIONS} reflections each")
print()

!python ai_scientist/perform_ideation_temp_free.py \
    --workshop-file "{topic}" \
    --model {IDEATION_MODEL} \
    --max-num-generations {NUM_IDEAS} \
    --num-reflections {NUM_REFLECTIONS} \
    2>&1

print("\n" + "─" * 40)
print("✅ Ideation complete!")

# Check the generated ideas file
ideas_file = PROJECT_DIR / ideas_out
if ideas_file.exists():
    with open(ideas_file) as f:
        ideas = json.load(f)
    if isinstance(ideas, list):
        print(f"\n📋 Generated {len(ideas)} ideas:")
        for i, idea in enumerate(ideas):
            print(f"   {i+1}. {idea.get('Title', idea.get('title', 'Untitled'))}")
    else:
        print(f"   ✅ Ideas saved to {ideas_file}")
else:
    print(f"   ⚠️ Ideas file not found at {ideas_file}")
    print("   Check the output above for errors.")

---
## 4. 🧪 Run Main Experiment Pipeline

This launches the AI-Scientist-v2 agentic tree search:
- **Stage 1:** Initial code generation & experiments
- **Stage 2-4:** Iterative improvement & debugging
- **Write-up:** Generates a LaTeX paper with plots
- **Review:** LLM self-review of the paper

> ⏱️ **Expected time:** 2-10+ hours depending on complexity and `steps` config.
> 
> 💾 **Checkpoints:** Results are saved automatically to `experiments/` folder.

In [ ]:
%cd {PROJECT_DIR}

# Determine which ideas file to use
if USE_EXAMPLE_IDEA:
    ideas_file = PROJECT_DIR / "ai_scientist" / "ideas" / "i_cant_believe_its_not_better.json"
else:
    ideas_file = PROJECT_DIR / "ai_scientist" / "ideas" / "my_research_topic.json"

if not ideas_file.exists():
    print(f"❌ No ideas file found at {ideas_file}!")
    print("   Run the ideation step first.")
else:
    print(f"✅ Using ideas file: {ideas_file}")
    
    # Quick preview
    with open(ideas_file) as f:
        ideas_data = json.load(f)
    if isinstance(ideas_data, list):
        print(f"   Found {len(ideas_data)} idea(s)")
        for i, idea in enumerate(ideas_data):
            print(f"   Idea {i}: {idea.get('Title', idea.get('title', 'N/A'))}")
    else:
        print(f"   Loaded idea data")

# ── Force bfts_config.yaml to use DeepSeek V4 Flash ──
print("\n⚙️ Ensuring bfts_config.yaml uses deepseek-v4-flash...")
config_path = PROJECT_DIR / "bfts_config.yaml"
config_text = config_path.read_text()
config_text = config_text.replace(
    "model: anthropic.claude-3-5-sonnet-20241022-v2:0",
    "model: deepseek-v4-flash"
)
config_text = config_text.replace("model: gpt-4o-2024-11-20", "model: deepseek-v4-flash")
config_path.write_text(config_text)
print("   ✅ Config updated to use deepseek-v4-flash for all stages")
print("   ✅ stage1_max_iters is set to 30 (from config)")

In [ ]:
%cd {PROJECT_DIR}

# ── Configure Models ──
# 🚀 deepseek-v4-pro for experiments, flash for writeup

EXPERIMENT_MODEL = "deepseek-v4-pro"     # Best for coding experiments 💪
WRITEUP_MODEL = "deepseek-v4-flash"      # Cheap for paper writing
REVIEW_MODEL = "deepseek-v4-flash"       # Cheap for review
CITATION_MODEL = "deepseek-v4-flash"     # Cheap for citations
PLOT_MODEL = "deepseek-v4-flash"         # Cheap for plot aggregation

# Use the correct ideas file
if USE_EXAMPLE_IDEA:
    IDEAS_PATH = "ai_scientist/ideas/i_cant_believe_its_not_better.json"
else:
    IDEAS_PATH = "ai_scientist/ideas/my_research_topic.json"

IDEA_IDX = 0                       # Which idea index to run (0 = first idea)
NUM_CITE_ROUNDS = 5                # Citation rounds (reduce for faster runs)

print("=" * 60)
print("🚀 Launching AI-Scientist-v2 Pipeline")
print("=" * 60)
print(f"📌 Topic:            Android Cybersecurity via AI")
print(f"📌 Ideas file:       {IDEAS_PATH}")
print(f"📌 Experiment model: {EXPERIMENT_MODEL}  ← Pro for code")
print(f"📌 Write-up model:   {WRITEUP_MODEL}     ← Flash for text")
print(f"📌 Idea index:       {IDEA_IDX}")
print(f"📌 Citation rounds:  {NUM_CITE_ROUNDS}")
print(f"📌 Kaggle optimized: workers=2, steps=3, drafts=1")
print(f"📌 Starter code:     --load_code enabled")
print("=" * 60)

# ── Launch the pipeline ──
# --load_code: uses my_research_topic.py as starter code
# --add_dataset_ref: includes dataset loading reference
!python launch_scientist_bfts.py \
    --load_ideas "{IDEAS_PATH}" \
    --load_code \
    --add_dataset_ref \
    --idea_idx {IDEA_IDX} \
    --model_writeup {WRITEUP_MODEL} \
    --model_citation {CITATION_MODEL} \
    --model_review {REVIEW_MODEL} \
    --model_agg_plots {PLOT_MODEL} \
    --num_cite_rounds {NUM_CITE_ROUNDS} \
    2>&1

print("\n" + "=" * 60)
print("✅ Pipeline finished!")
print("=" * 60)

---
## 5. 📊 Results & Export

After the pipeline completes, find your results in the `experiments/` folder.

### Expected output structure:
```
experiments/
  └── TIMESTAMP_IDEANAME/
      ├── TIMESTAMP_IDEANAME.pdf       ← 📄 Final paper!
      ├── logs/
      │   └── 0-run/
      │       └── unified_tree_viz.html ← 🌳 Tree visualization
      ├── workspaces/                    ← Experiment code & data
      └── token_tracker.json             ← 💰 Token usage
```

In [ ]:
%cd {PROJECT_DIR}

import glob

print("=" * 60)
print("📂 Results Summary")
print("=" * 60)

# ── Find generated PDFs ──
pdfs = list(PROJECT_DIR.glob("experiments/**/*.pdf"))
if pdfs:
    print(f"\n📄 Generated PDFs ({len(pdfs)} found):")
    for pdf in sorted(pdfs):
        size_mb = pdf.stat().st_size / 1e6
        print(f"   ✅ {pdf.relative_to(PROJECT_DIR)} ({size_mb:.1f} MB)")
else:
    print("\n   No PDFs found yet.")

# ── Find tree visualizations ──
viz_files = list(PROJECT_DIR.glob("experiments/**/unified_tree_viz.html"))
if viz_files:
    print(f"\n🌳 Tree visualizations ({len(viz_files)} found):")
    for viz in sorted(viz_files):
        print(f"   ✅ {viz.relative_to(PROJECT_DIR)}")

# ── Find experiments folder ──
exp_dirs = [d for d in (PROJECT_DIR / "experiments").iterdir() if d.is_dir()] if (PROJECT_DIR / "experiments").exists() else []
if exp_dirs:
    print(f"\n📁 Experiment directories ({len(exp_dirs)} found):")
    for d in sorted(exp_dirs):
        print(f"   📁 {d.name}")
        # Check for sub-items
        items = list(d.iterdir())
        for item in items[:10]:
            tag = "📁" if item.is_dir() else "📄"
            print(f"      {tag} {item.name}")
        if len(items) > 10:
            print(f"      ... and {len(items)-10} more items")

# ── Show token usage ──
token_files = list(PROJECT_DIR.glob("experiments/**/token_tracker.json"))
if token_files:
    print(f"\n💰 Token usage:")
    for tf in sorted(token_files):
        try:
            with open(tf) as f:
                tokens = json.load(f)
            print(f"   {tf.parent.name}: {json.dumps(tokens, indent=2)}")
        except:
            pass

print("\n" + "─" * 40)
print("💡 Tip: Download results from Kaggle")
print("   Go to the Data tab on the right → 'Output' → Download all")

---
## 6. 🚀 Optimizations & Tips

### Kaggle-optimized defaults (💰 ~$5-8 per run):
| Setting | Value | Why |
|---------|-------|-----|
| `num_workers` | `2` | ⬇️ from 4 (less RAM, more stable) |
| `steps` | `3` | ⬇️ from 5 (faster completion) |
| `num_drafts` | `1` | ⬇️ from 3 (single exploration path) |
| `num_seeds` | `2` | Matches num_workers |
| `stage1_max_iters` | `30` | ⬆️ from 20 (more attempts to find working code) |
| `--load_code` | ✅ enabled | Provides starter code (MLP baseline) |
| `--add_dataset_ref` | ✅ enabled | Adds dataset loading template |

### 💡 Optimization strategy:
1. **Starter code**: `my_research_topic.py` provides an MLP baseline — the AI iterates on it instead of writing from scratch
2. **`--resume` flag**: Reprendre après un échec sans refaire les expériences
3. **Cache ideation**: Les idées sont sauvegardées dans `.json` — réutilisables sans refaire l'idéation

### For higher quality (💰 $15-25 per run):
| Setting | Recommended |
|---------|-------------|
| Experiment model | `claude-3-5-sonnet-20241022` |
| `num_workers` | `4` |
| `steps` | `5` |
| `num_drafts` | `3` |

### Kaggle limits:
- **30h/semaine** GPU (free tier)
- **~9h** inactivity timeout
- Résultats dans `/kaggle/working/` (persistent)

In [ ]:
%cd {PROJECT_DIR}

# 🔄 RESUME MODE - Skip experiments, redo plotting + writeup
# ✏️ Replace with your actual experiment folder path:
RESUME_DIR = "experiments/2026-07-14_17-33-19_compositional_regularization_nn_attempt_0"

from pathlib import Path
if Path(RESUME_DIR).exists():
    print(f"✅ Found: {RESUME_DIR}")
    print("🚀 Resuming from plotting stage...")
else:
    print(f"❌ Not found: {RESUME_DIR}")
    print("   Available experiments:")
    for d in sorted(Path("experiments").iterdir()):
        if d.is_dir():
            print(f"   📁 {d.name}")
    RESUME_DIR = None

if RESUME_DIR:
    !python launch_scientist_bfts.py \
        --resume "{RESUME_DIR}" \
        --model_writeup deepseek-v4-flash \
        --model_citation deepseek-v4-flash \
        --model_review deepseek-v4-flash \
        --model_agg_plots deepseek-v4-flash \
        --num_cite_rounds 5 \
        2>&1

---
## 📖 Additional Resources

- 📚 [AI-Scientist-v2 Paper](https://pub.sakana.ai/ai-scientist-v2/paper)
- 🐙 [GitHub Repository](https://github.com/SakanaAI/AI-Scientist_v2)
- 📝 [Blog Post](https://sakana.ai/ai-scientist-first-publication/)
- ❓ [FAQ](https://github.com/SakanaAI/AI-Scientist_v2#frequently-asked-questions)

---
*Created for Kaggle - AI-Scientist-v2 Automated Scientific Discovery*